In [1]:
import pandas as pd
import json

In [2]:
# Načtení datasetu
df = pd.read_csv("../data/AirQualityUCI.csv", sep=';', decimal=',')

# Odstranění prázdných sloupců
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Odstranění prázdných řádků
df.dropna(how='all', inplace=True)

df.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


In [3]:
# Vynechání nečíselných sloupců (Date a Time)
data_cols = df.columns.difference(['Date', 'Time'])

In [4]:
# Vytvoření binární masky s chybějícími hodnotami: 1 (chybějící), 0 (úplná)
binary_mask = (df[data_cols] == -200).astype(int)

In [5]:
# Vytvoření řádkových vzorců z binární masky (jako tuple) a spočítání četnosti jednotlivých vzorů
pattern_counts = binary_mask.apply(lambda row: tuple(row), axis=1).value_counts()

# Zobrazení 10 nejčastějších vzorců chybějících hodnot
pattern_counts.head(10)

(0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0)    6114
(0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0)    1195
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)     827
(0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0)     428
(0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0)     364
(1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1)     291
(0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0)      36
(1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1)      31
(1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1)      26
(0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)      24
Name: count, dtype: int64

In [6]:
# Vytvoření výstupního seznamu z 10 nejčastějších vzorců
pattern_json = []

for pattern, count in pattern_counts.head(10).items():
    missing_features = [col for col, flag in zip(data_cols, pattern) if flag == 1]
    pattern_json.append({
        'missing_columns': missing_features,
        'frequency': int(count)
    })

# Uložení do JSON souboru
with open('../missing/export_patterns/missing_patterns.json', 'w', encoding='utf-8') as f:
    json.dump(pattern_json, f, ensure_ascii=False, indent=4)

In [7]:
# Načtení původního JSON se vzory
with open('../missing/export_patterns/missing_patterns.json', 'r', encoding='utf-8') as f:
    pattern_json = json.load(f)

# Filtrování vzorů, kde chybí více než jeden sloupec
grouped_patterns = [
    {
        'missing_group': p['missing_columns'],
        'support': p['frequency']
    }
    for p in pattern_json
    if len(p['missing_columns']) >= 2
]

# Uložení do nového JSON souboru
with open('../missing/export_patterns/grouped_patterns.json', 'w', encoding='utf-8') as f:
    json.dump(grouped_patterns, f, ensure_ascii=False, indent=4)